# Chapter 20: ICS/OT and Critical Infrastructure Security

> "In operational technology, a security failure is not a data breach. It is a physical event."

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Define ICS, OT, PLC, and SCADA and explain how OT differs from IT.
2. Describe the Purdue Reference Model and the role of network segmentation.
3. Name the principal OT security standards and what each governs.
4. Explain why legacy industrial protocols are insecure by design.
5. Apply baseline hardening steps to an industrial control device.

## Key Terms

- **ICS**: Industrial Control System.
- **OT**: Operational Technology, the hardware and software that monitors and controls physical processes.
- **PLC**: Programmable Logic Controller, a ruggedized computer that controls machinery.
- **SCADA**: Supervisory Control and Data Acquisition.
- **HMI**: Human-Machine Interface.
- **Modbus / DNP3 / EtherNet-IP**: common industrial protocols.
- **Purdue Model**: a hierarchical reference model for segmenting industrial networks.
- **IEC 62443**: international standard series for industrial automation and control system security.

---

## 20.1 Why OT Is Different from IT

Information technology protects data; operational technology controls physical processes such as turbines,
valves, and assembly lines. The priorities invert. In IT the classic ordering is confidentiality,
integrity, availability. In OT it is typically availability and safety first, because stopping a process or
corrupting a control command can cause physical damage, environmental harm, or loss of life. OT systems also
have very long lifespans measured in decades, run software that cannot be patched without halting
production, and were historically designed for isolated networks where no authentication seemed necessary.

## 20.2 The Components

A **PLC** is a small ruggedized controller that reads sensors and drives actuators according to a program,
often written as ladder logic. Real families include the Siemens S7 and the Allen-Bradley line.
**SCADA** systems supervise many distributed controllers, gathering telemetry and issuing commands across a
plant or a region such as a pipeline or an electrical grid. An **HMI** is the operator's screen. Smart-building
devices such as HVAC controllers, networked locks, and IP cameras are increasingly part of the same attack
surface.

## 20.3 Insecure-by-Design Protocols

Many industrial protocols predate any expectation of network hostility. Modbus and DNP3 in their classic
forms carry no authentication and no encryption, so any device that can reach the network can issue
commands. This is why the Stuxnet campaign, which targeted PLC firmware to damage centrifuges, and the
TRITON malware, which targeted a safety-instrumented system, are studied as turning points: they
demonstrated that code can produce physical, safety-relevant consequences. Compromise of a control protocol
is not about reading data; it is about commanding equipment.

## 20.4 The Purdue Reference Model and Segmentation

The Purdue Reference Model organizes an industrial environment into hierarchical levels, from field devices
and sensors at the bottom (Levels 0 and 1), through control and supervisory networks (Levels 2 and 3), up to
enterprise IT (Levels 4 and 5). The central defensive idea is segmentation: traffic between levels passes
through controlled boundaries, and the control network should never be directly reachable from the internet.
Most real OT intrusions exploit a flat network or an unintended route from IT into OT rather than a novel
exploit against the controller itself.

## 20.5 Standards and Frameworks

- **NIST SP 800-82 Rev. 3** is the Guide to Operational Technology Security and covers OT-specific threat
  models, risk management, and defensive architecture {cite}`nist_sp800_82`.
- **IEC 62443** is the international series for industrial automation and control system security and defines
  security levels SL 0 through SL 4 for zones and conduits {cite}`iec_62443`.
- **NERC CIP** sets mandatory critical-infrastructure protection requirements for the North American bulk
  electric system.
- **MITRE ATT&CK for ICS** is an adversary technique knowledge base specific to ICS, mapping campaigns such
  as Stuxnet and TRITON to tactics and techniques {cite}`mitre_attack`.
- **CISA ICS advisories** publish newly discovered ICS vulnerabilities for defenders to track.

## 20.6 Why This Matters

Water treatment, power generation, manufacturing, and transportation all depend on OT. As these networks
connect to enterprise IT for efficiency, the attack surface that was once protected only by isolation becomes
reachable. A defender who understands both the IT techniques in the rest of this book and the physical-first
priorities of OT is positioned to protect the systems whose failure has the most direct human consequences.

## 20.7 News in Focus

Public advisories in recent years have repeatedly warned of intrusions into operational-technology
environments in the water and energy sectors, often exploiting internet-exposed devices left with default
credentials rather than sophisticated zero-day exploits. The recurring theme is that basic hygiene, removing
internet exposure and changing default passwords, would have blocked many documented incidents.

## 20.8 Worked Example: Auditing an ICS Hardening Baseline

The checker below evaluates a controller's configuration against a baseline drawn from standard OT hardening
guidance: network isolation, disabled legacy services, strong credentials, and monitoring.


In [1]:
def audit_plc(config):
    findings = []
    if config.get("internet_routable", True):
        findings.append("CRITICAL: controller is internet-routable; isolate on a dedicated VLAN")
    if config.get("default_password", True):
        findings.append("CRITICAL: default web-interface password still set; change it")
    for svc in ("telnet", "ftp", "http"):
        if config.get(svc, False) and not config.get(f"{svc}_required", False):
            findings.append(f"HIGH: {svc} enabled but not required; disable it")
    if not config.get("mfa_on_workstation", False):
        findings.append("MEDIUM: no MFA on the engineering workstation")
    if not config.get("traffic_monitoring", False):
        findings.append("MEDIUM: no baseline monitoring of control-protocol traffic")
    if not config.get("logic_backup", False):
        findings.append("LOW: control logic is not backed up offline with an integrity checksum")
    return findings

example = {
    "internet_routable": False,
    "default_password": True,
    "telnet": True, "telnet_required": False,
    "ftp": False, "http": True, "http_required": True,
    "mfa_on_workstation": False,
    "traffic_monitoring": True,
    "logic_backup": False,
}

results = audit_plc(example)
print(f"{len(results)} finding(s):")
for f in results:
    print(" -", f)


4 finding(s):
 - CRITICAL: default web-interface password still set; change it
 - HIGH: telnet enabled but not required; disable it
 - MEDIUM: no MFA on the engineering workstation
 - LOW: control logic is not backed up offline with an integrity checksum


## 20.9 Review Questions (MCQ)

**Q1.** In operational technology the security priority that typically comes first is:
A. Confidentiality  B. Availability and safety  C. Non-repudiation  D. Auditability

**Q2.** The primary defensive idea of the Purdue Reference Model is:
A. Encryption of all traffic  B. Hierarchical segmentation between levels  C. Multi-factor authentication  D. Antivirus on PLCs

**Q3.** Classic Modbus is considered insecure mainly because it:
A. Is too slow  B. Lacks authentication and encryption  C. Uses too much bandwidth  D. Is proprietary

*Answers: Q1 B, Q2 B, Q3 B.*

## 20.10 Lab Assignment

Take the audit function above and extend the baseline with at least three additional checks drawn from
NIST SP 800-82 or IEC 62443 (for example firmware-version verification or removal of shared accounts). Then
write a short remediation plan that orders the findings by severity and explains the physical consequence of
leaving each one unaddressed.

## References

```{bibliography}
:filter: docname in docnames
```
